**This notebook demonstrates data ingestion and reading workflows using PySpark. The primary objective is to show how different types of data files (such as CSV, JSON, and Parquet. etc...) can be loaded into Spark DataFrames for further processing and analysis.**

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import pandas as pd
import pprint

Mounting external storage (Google Drive) to access datasets.

In [ ]:
def source_mount():
    """
    Mount Google Drive in Google Colab.

    This function mounts Google Drive to '/content/drive' in Colab.
    It uses try-except to handle errors gracefully.
    No parameters are required.
    """
    try:
        from google.colab import drive
        drive.mount("/content/drive/")
        return "Drive mounted successfully. :)"
    except Exception as e:
        return f"Error occurred during drive mounting:( {e}"

print(source_mount())

Mounted at /content/drive/
Drive mounted successfully. :)


Initializing a Spark session in a Colab/Notebook environment.

In [3]:
spark = \
 (
    SparkSession.builder
    .appName("DataIngestionDemo")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "15")    # Controls parallelism in shuffles
    .config("spark.executor.memory", "2g")           # Memory per executor
    .config("spark.driver.memory", "2g")             # Memory for driver
    .config("spark.sql.session.timeZone", "UTC")     # Set timezone for timestamps
    .config("spark.jars", "sqlite-jdbc-3.36.0.3.jar")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")  # Speed up Pandas conversion
    .getOrCreate()
 )

# Check active configs
for item in spark.sparkContext.getConf().getAll():
    print(item)


('spark.hadoop.fs.s3a.vectored.read.min.seek.size', '128K')
('spark.driver.memory', '2g')
('spark.executor.extraJavaOptions', '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectM

**Ingestion of *`CSV`* Data with Defined Data Types in PySpark DataFrames**


In [33]:
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "utf-8")
    .csv("/content/cust_info.csv")
)

print("CSV Data Load Successfully :)")

CSV Data Load Successfully :)


In [34]:
df.printSchema()

root
 |-- cst_id: integer (nullable = true)
 |-- cst_key: string (nullable = true)
 |-- cst_firstname: string (nullable = true)
 |-- cst_lastname: string (nullable = true)
 |-- cst_marital_status: string (nullable = true)
 |-- cst_gndr: string (nullable = true)
 |-- cst_create_date: date (nullable = true)



In [35]:
df.summary().show()

+-------+------------------+-----------+-------------+------------+------------------+--------+
|summary|            cst_id|    cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|
+-------+------------------+-----------+-------------+------------+------------------+--------+
|  count|             18490|      18494|        18486|       18487|             18487|   13916|
|   mean|20244.491941590048|1.3451235E7|         NULL|        NULL|              NULL|    NULL|
| stddev| 5337.733647606977|       NULL|         NULL|        NULL|              NULL|    NULL|
|    min|             11000|   13451235|        Chloe|       Brown|                 M|       F|
|    25%|             15620|1.3451235E7|         NULL|        NULL|              NULL|    NULL|
|    50%|             20242|1.3451235E7|         NULL|        NULL|              NULL|    NULL|
|    75%|             24865|1.3451235E7|         NULL|        NULL|              NULL|    NULL|
|    max|             29483|      SF566|

In [36]:
df.count()

18494

In [37]:
df.show(30, truncate=False)

+------+----------+-------------+------------+------------------+--------+---------------+
|cst_id|cst_key   |cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|
+------+----------+-------------+------------+------------------+--------+---------------+
|11000 |AW00011000| Jon         |Yang        |M                 |M       |2025-10-06     |
|11001 |AW00011001|Eugene       |Huang       |S                 |M       |2025-10-06     |
|11002 |AW00011002|Ruben        | Torres     |M                 |M       |2025-10-06     |
|11003 |AW00011003|Christy      |  Zhu       |S                 |F       |2025-10-06     |
|11004 |AW00011004| Elizabeth   |Johnson     |S                 |F       |2025-10-06     |
|11005 |AW00011005|Julio        |Ruiz        |S                 |M       |2025-10-06     |
|11006 |AW00011006|Janet        |Alvarez     |S                 |F       |2025-10-06     |
|11007 |AW00011007|Marco        |Mehta       |M                 |M       |2025-10-06     |

**Ingestion of *`Parquet`* Data with Defined Data Types in PySpark DataFrames**





In [38]:
df = (
    spark.read
    .parquet("/content/sales_details.parquet")
)

print("Parquet Data Load Successfully :)")

Parquet Data Load Successfully :)


In [39]:
df.schema

StructType([StructField('sls_ord_num', StringType(), True), StructField('sls_prd_key', StringType(), True), StructField('sls_cust_id', IntegerType(), True), StructField('sls_order_dt', IntegerType(), True), StructField('sls_ship_dt', IntegerType(), True), StructField('sls_due_dt', IntegerType(), True), StructField('sls_sales', IntegerType(), True), StructField('sls_quantity', IntegerType(), True), StructField('sls_price', IntegerType(), True)])

In [40]:
df.dtypes

[('sls_ord_num', 'string'),
 ('sls_prd_key', 'string'),
 ('sls_cust_id', 'int'),
 ('sls_order_dt', 'int'),
 ('sls_ship_dt', 'int'),
 ('sls_due_dt', 'int'),
 ('sls_sales', 'int'),
 ('sls_quantity', 'int'),
 ('sls_price', 'int')]

In [41]:
df.sample(fraction=0.1, withReplacement=False).show(30)

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|    SO43708| BK-R50R-52|      20042|    20101231|   20110107|  20110112|      699|           1|      699|
|    SO43730| BK-M82S-44|      25861|    20110106|   20110113|  20110118|     3400|           1|     3400|
|    SO43736| BK-M82S-44|      11002|    20110107|   20110114|  20110119|     3400|           1|     3400|
|    SO43745| BK-R93R-44|      16514|    20110110|   20110117|  20110122|     3578|           1|     3578|
|    SO43746| BK-R93R-48|      16616|    20110110|   20110117|  20110122|     3578|           1|     3578|
|    SO43756| BK-R50B-58|      19941|    20110112|   20110119|  20110124|      699|           1|      699|
|    SO43784| BK-R93R-62|      27667|

**Ingestion of *`ORC`* Data with Defined Data Types in PySpark DataFrames**

In [42]:
df = (
    spark.read
    .orc("/content/cust_info.orc")
)

print("ORC Data Load Successfully :)")

ORC Data Load Successfully :)


In [43]:
df.show(30)

+------+----------+-------------+------------+------------------+--------+---------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|
+------+----------+-------------+------------+------------------+--------+---------------+
| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|
| 11001|AW00011001|       Eugene|     Huang  |                 S|       M|     2025-10-06|
| 11002|AW00011002|        Ruben|      Torres|                 M|       M|     2025-10-06|
| 11003|AW00011003|      Christy|         Zhu|                 S|       F|     2025-10-06|
| 11004|AW00011004|    Elizabeth|     Johnson|                 S|       F|     2025-10-06|
| 11005|AW00011005|        Julio|        Ruiz|                 S|       M|     2025-10-06|
| 11006|AW00011006|        Janet|     Alvarez|                 S|       F|     2025-10-06|
| 11007|AW00011007|        Marco|       Mehta|                 M|       M|     2025-10-06|

In [ ]:
df.describe().show()

+-------+------------------+-----------+-------------+------------+------------------+--------+
|summary|            cst_id|    cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|
+-------+------------------+-----------+-------------+------------+------------------+--------+
|  count|             18490|      18494|        18486|       18487|             18487|   13916|
|   mean|20244.491941590048|1.3451235E7|         NULL|        NULL|              NULL|    NULL|
| stddev| 5337.733647606977|       NULL|         NULL|        NULL|              NULL|    NULL|
|    min|             11000|   13451235|        Chloe|       Brown|                 M|       F|
|    max|             29483|      SF566|          Zoe|    Zukowski|                 S|       M|
+-------+------------------+-----------+-------------+------------+------------------+--------+



In [ ]:
df.dtypes

[('cst_id', 'int'),
 ('cst_key', 'string'),
 ('cst_firstname', 'string'),
 ('cst_lastname', 'string'),
 ('cst_marital_status', 'string'),
 ('cst_gndr', 'string'),
 ('cst_create_date', 'date')]

**Ingestion of *`Json`* Data with Defined Data Types in PySpark DataFrames**

In [13]:
df = (
    spark.read
    .option("encoding", "utf-8")
    .option("multiLine", "true")
    .json("/content/sample_data/anscombe.json")
)

print("Json Data Load Successfully :)")

Json Data Load Successfully :)


In [14]:
df.show()

+------+----+-----+
|Series|   X|    Y|
+------+----+-----+
|     I|10.0| 8.04|
|     I| 8.0| 6.95|
|     I|13.0| 7.58|
|     I| 9.0| 8.81|
|     I|11.0| 8.33|
|     I|14.0| 9.96|
|     I| 6.0| 7.24|
|     I| 4.0| 4.26|
|     I|12.0|10.84|
|     I| 7.0| 4.81|
|     I| 5.0| 5.68|
|    II|10.0| 9.14|
|    II| 8.0| 8.14|
|    II|13.0| 8.74|
|    II| 9.0| 8.77|
|    II|11.0| 9.26|
|    II|14.0|  8.1|
|    II| 6.0| 6.13|
|    II| 4.0|  3.1|
|    II|12.0| 9.13|
+------+----+-----+
only showing top 20 rows


In [ ]:
df.count()

44

In [ ]:
df.columns

['Series', 'X', 'Y']

In [15]:
jdf = df.withColumnsRenamed \
 (
    {
       "Series" : "Series_Number",
       "X"      : "x-axis",
       "Y"      : "y-axis"
    }
)

In [16]:
jdf.show()

+-------------+------+------+
|Series_Number|x-axis|y-axis|
+-------------+------+------+
|            I|  10.0|  8.04|
|            I|   8.0|  6.95|
|            I|  13.0|  7.58|
|            I|   9.0|  8.81|
|            I|  11.0|  8.33|
|            I|  14.0|  9.96|
|            I|   6.0|  7.24|
|            I|   4.0|  4.26|
|            I|  12.0| 10.84|
|            I|   7.0|  4.81|
|            I|   5.0|  5.68|
|           II|  10.0|  9.14|
|           II|   8.0|  8.14|
|           II|  13.0|  8.74|
|           II|   9.0|  8.77|
|           II|  11.0|  9.26|
|           II|  14.0|   8.1|
|           II|   6.0|  6.13|
|           II|   4.0|   3.1|
|           II|  12.0|  9.13|
+-------------+------+------+
only showing top 20 rows


**Ingestion of Database(.db) Data with Defined Data Types in PySpark DataFrames**

In [ ]:
import sqlite3

conn = sqlite3.connect("/content/rides.db")
cursor = conn.cursor()

# Get column info for 'rides' table
cursor.execute("PRAGMA table_info(rides);")
columns = cursor.fetchall()

# Print column names
print("Columns in rides table:")
for col in columns:
    print(col[1], "-", col[2])   # col[1] = column name, col[2] = data type

conn.close()


Columns in rides table:
vendor - TEXT
count - INTEGER
pickup - TIMESTAMP
dropoff - TIMESTAMP
distance - REAL
tip - REAL
total - REAL


In [19]:
import sqlite3

# Connect to the .db file
conn = sqlite3.connect("/content/rides.db")
cursor = conn.cursor()

# Query to list all tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Tables inside rides.db:", tables)


Tables inside rides.db: [('rides',)]


In [20]:
import sqlite3

conn = sqlite3.connect("/content/rides.db")
cursor = conn.cursor()

# Run query
cursor.execute("SELECT * FROM rides")

# Fetch all rows
rows = cursor.fetchall()

# Print first 5 rows
for row in rows[:30]:
    print(row)

conn.close()


('VeriFone', 1, '2018-10-31 23:27:39', '2018-10-31 23:44:33', 7.91, 0.0, 24.3)
('VeriFone', 1, '2018-11-01 00:00:17', '2018-11-01 00:20:16', 3.23, 2.0, 18.8)
('VeriFone', 2, '2018-11-01 00:00:42', '2018-11-01 00:04:29', 1.46, 0.0, 7.3)
('Creative', 2, '2018-11-01 00:02:06', '2018-11-01 00:26:27', 3.5, 3.85, 23.15)
('Creative', 1, '2018-11-01 00:02:29', '2018-11-01 00:04:32', 0.4, 0.95, 5.75)
('Creative', 1, '2018-11-01 00:02:45', '2018-11-01 00:07:43', 0.7, 0.0, 6.3)
('VeriFone', 1, '2018-11-01 00:03:04', '2018-11-01 00:16:29', 2.74, 3.84, 16.64)
('VeriFone', 2, '2018-11-01 00:03:46', '2018-11-01 00:09:58', 0.84, 0.0, 7.3)
('Creative', 1, '2018-11-01 00:05:43', '2018-11-01 00:26:57', 4.6, 3.0, 22.3)
('VeriFone', 3, '2018-11-01 00:06:25', '2018-11-01 00:34:35', 2.94, 3.86, 23.16)
('Creative', 1, '2018-11-01 00:06:43', '2018-11-01 00:30:23', 1.9, 2.45, 18.75)
('VeriFone', 1, '2018-11-01 00:07:22', '2018-11-01 00:21:35', 3.91, 3.06, 18.36)
('Creative', 1, '2018-11-01 00:08:03', '2018-11-0

In [21]:
import sqlite3

# Spark session
spark = SparkSession.builder.appName("SQLiteToSpark").getOrCreate()

# Connect to SQLite
conn = sqlite3.connect("/content/rides.db")
cursor = conn.cursor()

# Read full table into Pandas DataFrame directly
pdf = pd.read_sql_query("SELECT * FROM rides", conn)
conn.close()

# Convert Pandas DataFrame to Spark DataFrame
df = spark.createDataFrame(pdf)

In [22]:
df.printSchema()

root
 |-- vendor: string (nullable = true)
 |-- count: long (nullable = true)
 |-- pickup: string (nullable = true)
 |-- dropoff: string (nullable = true)
 |-- distance: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- total: double (nullable = true)



In [23]:
df.show()

+--------+-----+-------------------+-------------------+--------+----+-----+
|  vendor|count|             pickup|            dropoff|distance| tip|total|
+--------+-----+-------------------+-------------------+--------+----+-----+
|VeriFone|    1|2018-10-31 23:27:39|2018-10-31 23:44:33|    7.91| 0.0| 24.3|
|VeriFone|    1|2018-11-01 00:00:17|2018-11-01 00:20:16|    3.23| 2.0| 18.8|
|VeriFone|    2|2018-11-01 00:00:42|2018-11-01 00:04:29|    1.46| 0.0|  7.3|
|Creative|    2|2018-11-01 00:02:06|2018-11-01 00:26:27|     3.5|3.85|23.15|
|Creative|    1|2018-11-01 00:02:29|2018-11-01 00:04:32|     0.4|0.95| 5.75|
|Creative|    1|2018-11-01 00:02:45|2018-11-01 00:07:43|     0.7| 0.0|  6.3|
|VeriFone|    1|2018-11-01 00:03:04|2018-11-01 00:16:29|    2.74|3.84|16.64|
|VeriFone|    2|2018-11-01 00:03:46|2018-11-01 00:09:58|    0.84| 0.0|  7.3|
|Creative|    1|2018-11-01 00:05:43|2018-11-01 00:26:57|     4.6| 3.0| 22.3|
|VeriFone|    3|2018-11-01 00:06:25|2018-11-01 00:34:35|    2.94|3.86|23.16|

**Ingestion of *`Text`* Data with Defined Data Types in PySpark DataFrames**

In [24]:
df = (
    spark.read
    .text("/content/raw_employees.txt")
)

print("CSV Data Load Successfully :)")

CSV Data Load Successfully :)


In [25]:
df.show(30, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                  |
+---------------------------------------------------------------------------------------------------------------------------------------+
|customer_key  customer_id  customer_number  first_name  last_name   country        marital_status  gender      birth_date  created_date|
|------------  -----------  ---------------  ----------  ----------  -------------  --------------  ----------  ----------  ------------|
|1             11000        AW00011000       Jon         Yang        Australia      Married         Male        1971-10-06  2025-10-06  |
|2             11001        AW00011001       Eugene      Huang       Australia      Single          Male        1976-05-10  2025-10-06  |
|3             11002        AW0001

**Ingestion of *`Excel`* *`xlsx`* Data with Defined Data Types in PySpark DataFrames**

In [26]:
pdf = pd.read_excel("/content/sample_100x8_dataset.xlsx")
df = spark.createDataFrame(pdf)

In [27]:
df.show(30, truncate=False)

+--------+--------+--------+--------+--------+--------+--------+--------+
|Column_1|Column_2|Column_3|Column_4|Column_5|Column_6|Column_7|Column_8|
+--------+--------+--------+--------+--------+--------+--------+--------+
|R1_C1   |R1_C2   |R1_C3   |R1_C4   |R1_C5   |R1_C6   |R1_C7   |R1_C8   |
|R2_C1   |R2_C2   |R2_C3   |R2_C4   |R2_C5   |R2_C6   |R2_C7   |R2_C8   |
|R3_C1   |R3_C2   |R3_C3   |R3_C4   |R3_C5   |R3_C6   |R3_C7   |R3_C8   |
|R4_C1   |R4_C2   |R4_C3   |R4_C4   |R4_C5   |R4_C6   |R4_C7   |R4_C8   |
|R5_C1   |R5_C2   |R5_C3   |R5_C4   |R5_C5   |R5_C6   |R5_C7   |R5_C8   |
|R6_C1   |R6_C2   |R6_C3   |R6_C4   |R6_C5   |R6_C6   |R6_C7   |R6_C8   |
|R7_C1   |R7_C2   |R7_C3   |R7_C4   |R7_C5   |R7_C6   |R7_C7   |R7_C8   |
|R8_C1   |R8_C2   |R8_C3   |R8_C4   |R8_C5   |R8_C6   |R8_C7   |R8_C8   |
|R9_C1   |R9_C2   |R9_C3   |R9_C4   |R9_C5   |R9_C6   |R9_C7   |R9_C8   |
|R10_C1  |R10_C2  |R10_C3  |R10_C4  |R10_C5  |R10_C6  |R10_C7  |R10_C8  |
|R11_C1  |R11_C2  |R11_C3  |R11_C4  |R

In [ ]:
print(df.columns)

['Column_1', 'Column_2', 'Column_3', 'Column_4', 'Column_5', 'Column_6', 'Column_7', 'Column_8']


In [ ]:
df.summary().show()

+-------+--------+--------+--------+--------+--------+--------+--------+--------+
|summary|Column_1|Column_2|Column_3|Column_4|Column_5|Column_6|Column_7|Column_8|
+-------+--------+--------+--------+--------+--------+--------+--------+--------+
|  count|     100|     100|     100|     100|     100|     100|     100|     100|
|   mean|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
| stddev|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|    min| R100_C1| R100_C2| R100_C3| R100_C4| R100_C5| R100_C6| R100_C7| R100_C8|
|    25%|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|    50%|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|    75%|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|    max|   R9_C1|   R9_C2|   R9_C3|   R9_C4|   R9_C5|   R9_C6|   R9_C7|   R9_C8|
+-------+--------+--------+--------+--------+--------+--------+--------+--------+



**Ingestion of  Data with Defined Data Types in PySpark DataFrames**

In [28]:
import xml.etree.ElementTree as ET
import pandas as pd

tree = ET.parse("/content/customers.xml")
root = tree.getroot()

rows = []
for row in root.findall(".//{http://schemas.openxmlformats.org/spreadsheetml/2006/main}row"):
    values = []
    for c in row.findall("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}c"):
        is_elem = c.find("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}is")
        v_elem = c.find("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}v")
        if is_elem is not None:
            values.append(is_elem.find("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t").text)
        elif v_elem is not None:
            values.append(v_elem.text)
        else:
            values.append(None)
    rows.append(values)

# Convert to Pandas DataFrame
pdf = pd.DataFrame(rows)

# First row as header
pdf.columns = pdf.iloc[0]   # set first row as header
pdf = pdf.drop(0)           # drop header row from data

# Convert to Spark DataFrame
df = spark.createDataFrame(pdf)

In [ ]:
df.printSchema()

root
 |-- customer_key: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_number: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- created_date: string (nullable = true)



In [29]:
df.show(30, truncate=False)

+------------+-----------+---------------+----------+---------+-------------+--------------+------+----------+------------+
|customer_key|customer_id|customer_number|first_name|last_name|country      |marital_status|gender|birth_date|created_date|
+------------+-----------+---------------+----------+---------+-------------+--------------+------+----------+------------+
|1           |11000      |AW00011000     |Jon       |Yang     |Australia    |Married       |Male  |26212     |45936       |
|2           |11001      |AW00011001     |Eugene    |Huang    |Australia    |Single        |Male  |27890     |45936       |
|3           |11002      |AW00011002     |Ruben     |Torres   |Australia    |Married       |Male  |25973     |45936       |
|4           |11003      |AW00011003     |Christy   |Zhu      |Australia    |Single        |Female|26890     |45936       |
|5           |11004      |AW00011004     |Elizabeth |Johnson  |Australia    |Single        |Female|29072     |45936       |
|6      

**Ingestion of *`Json Lin(.jl)`* Data with Defined Data Types in PySpark DataFrames**

In [30]:
df = (
    spark.read
    .option("multiline", "false")
    .json("/content/taxi.jl")
)

In [31]:
df.show(30 , truncate=False)

+--------+------------------------+------------------------+-----+-----+------+
|distance|dropoff                 |pickup                  |tip  |total|vendor|
+--------+------------------------+------------------------+-----+-----+------+
|2.57    |2018-11-01T06:43:24.000Z|2018-10-31T07:10:55.000Z|4.74 |20.54|2     |
|3.58    |2018-10-31T16:50:10.000Z|2018-10-31T16:38:25.000Z|0.0  |13.8 |2     |
|2.39    |2018-10-31T20:31:47.000Z|2018-10-31T20:23:41.000Z|1.0  |11.3 |2     |
|0.5     |2018-10-31T22:48:28.000Z|2018-10-31T22:44:24.000Z|0.0  |5.8  |2     |
|1.81    |2018-10-31T23:35:30.000Z|2018-10-31T23:22:18.000Z|2.26 |13.56|2     |
|7.91    |2018-10-31T23:44:33.000Z|2018-10-31T23:27:39.000Z|0.0  |24.3 |2     |
|2.24    |2018-10-31T23:46:52.000Z|2018-10-31T23:40:55.000Z|2.32 |11.62|2     |
|2.31    |2018-10-31T23:56:44.000Z|2018-10-31T23:45:53.000Z|0.0  |11.3 |2     |
|0.68    |2018-10-31T23:52:54.000Z|2018-10-31T23:46:42.000Z|0.0  |6.8  |2     |
|1.65    |2018-11-01T00:10:33.000Z|2018-

In [ ]:
df.printSchema()

root
 |-- distance: double (nullable = true)
 |-- dropoff: string (nullable = true)
 |-- pickup: string (nullable = true)
 |-- tip: double (nullable = true)
 |-- total: double (nullable = true)
 |-- vendor: long (nullable = true)

